# NeuralScene Bench

Controlled novel-view synthesis benchmark for Netflix JR40251 portfolio work.

Initial scope: Mip-NeRF 360 Bonsai, comparing Nerfstudio `splatfacto` and `nerfacto` under a shared Nerfstudio evaluation pipeline.

Important: results from this notebook should be compared only within this benchmark. Do not directly mix these metrics with the earlier gsplat FastGS or SplatStream numbers because the training and evaluation pipelines differ.

Pinned Nerfstudio source revision: `50e0e3c70c775e89333256213363badbf074f29d`.

## Step 1. Check the Colab GPU environment
Run this cell first.

In [ ]:
import sys
import subprocess
print('Python:', sys.version)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA build:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch check:', e)
print('\n--- NVIDIA SMI ---')
subprocess.run(['nvidia-smi'])

## Step 2. Install the pinned benchmark environment

The current Colab host uses Python 3.13. To reduce binary-wheel and CUDA-extension risk, this benchmark creates an isolated Python 3.11 environment instead of modifying the Colab kernel. Nerfstudio is pinned to one source revision and PyTorch is pinned to a CUDA 12.8 build.

Run this cell once per fresh Colab runtime.

In [ ]:
from pathlib import Path
import subprocess
import sys
NS_COMMIT = '50e0e3c70c775e89333256213363badbf074f29d'
VENV = Path('/content/neuralscene-env')
NS = Path('/content/nerfstudio')
PY = VENV / 'bin/python'
def run(cmd, cwd=None):
    print('\n$', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not PY.exists():
    run(['uv', 'venv', '--python', '3.11', VENV])
run(['uv', 'pip', 'install', '--python', PY, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu128'])
if not NS.exists():
    run(['git', 'clone', 'https://github.com/nerfstudio-project/nerfstudio.git', NS])
run(['git', 'fetch', '--all', '--tags'], cwd=NS)
run(['git', 'reset', '--hard', NS_COMMIT], cwd=NS)
run(['uv', 'pip', 'install', '--python', PY, '-e', NS])
run([PY, '-c', "import torch; print('Environment ready'); print('Python environment OK'); print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])
print('\nPinned Nerfstudio commit:')
run(['git', 'rev-parse', 'HEAD'], cwd=NS)

## Step 3. Prepare Mip-NeRF 360 Bonsai

This uses a scene-level Mip-NeRF 360 Bonsai mirror to avoid downloading the full multi-scene archive. The cell resumes an interrupted download when the Colab runtime itself has not been reset, extracts the scene, then prints the discovered COLMAP and image layout. No training starts here.

In [ ]:
from pathlib import Path
import subprocess, zipfile, shutil

URL = 'https://data.ciirc.cvut.cz/public/projects/2023NerfBaselines/data/gaussian-splatting/mipnerf360/bonsai.zip'
ZIP = Path('/content/bonsai_mipnerf360.zip')
ROOT = Path('/content/data/mipnerf360')
SCENE = ROOT / 'bonsai'
ROOT.mkdir(parents=True, exist_ok=True)

if not SCENE.exists():
    print('Downloading Bonsai scene archive. This is about 1.3 GB and wget -c can resume within the same runtime.', flush=True)
    subprocess.run(['wget', '-c', URL, '-O', str(ZIP)], check=True)
    tmp = ROOT / '_bonsai_extract'
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP, 'r') as zf:
        zf.extractall(tmp)
    candidates = [p for p in tmp.rglob('*') if p.is_dir() and ((p/'sparse').exists() or (p/'colmap').exists()) and any((p/q).exists() for q in ['images','images_2','images_4'])]
    if not candidates:
        raise RuntimeError('Could not locate a Bonsai dataset root after extraction.')
    src = sorted(candidates, key=lambda p: len(p.parts))[0]
    if SCENE.exists():
        shutil.rmtree(SCENE)
    shutil.move(str(src), str(SCENE))
    shutil.rmtree(tmp, ignore_errors=True)

print('Bonsai root:', SCENE)
print('Exists:', SCENE.exists())
print('Images:', [p.name for p in SCENE.iterdir() if p.is_dir() and p.name.startswith('images')])
print('sparse/0:', (SCENE/'sparse/0').exists())
print('colmap/sparse/0:', (SCENE/'colmap/sparse/0').exists())
print('Top-level:', sorted(p.name for p in SCENE.iterdir())[:30])


## Next

After Step 3 prints the Bonsai layout successfully, the next notebook update will add a short smoke test before the full Splatfacto and Nerfacto benchmark runs.